# Support Integrity Auditor (SIA)

Self-supervised auditor for **Priority Mismatch** in CRM support tickets. This notebook runs the full reproducible pipeline:

**Stage 1** data prep → **Stage 2** self-supervised pseudo-labeling (signal fusion + ablation) → **Stage 3** fine-tuned classifier → **inference + Evidence Dossier**.

There are no pre-annotated mismatch labels — the supervision signal is bootstrapped from raw ticket data by fusing independent severity signals and comparing the inferred severity to the human-assigned priority.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('.'))
from src import config as C
import pandas as pd
pd.set_option('display.max_colwidth', 90)

## Stage 1 — Data preparation
Parse the genuine issue sentence out of the faker-padded description, build the model text + structured features, and lock a stratified held-out split.

In [ ]:
from src import data_prep
df = data_prep.main()
df[[C.COL_PRIORITY, C.COL_CATEGORY, 'lead_sentence', 'model_text']].head()

## Stage 2 — Pseudo-label generation (self-supervised)
Fuse three independent severity signals — rule-based lexical, neural semantic (sentence-transformers), and resolution-time — into an inferred severity, then derive the binary mismatch label by comparing to the assigned priority. The ablation shows each signal's individual contribution.

In [ ]:
from src import pseudo_label
labeled = pseudo_label.main()
print(json.load(open(C.METRICS_DIR / 'pseudolabel_stats.json')))
pd.read_csv(C.METRICS_DIR / 'ablation.csv')

## Stage 3 — Fine-tune the binary classifier
Class-weighted fine-tuning of an MPS-stable encoder on `model_text` (assigned priority + channel/category/tier tags + the real issue sentence). Held-out verification against the thresholds (Accuracy ≥ 83%, Macro-F1 ≥ 0.82, per-class recall ≥ 0.78).

> This cell trains for several minutes. You can also run it from the shell: `python3 train_pipeline.py`.

In [ ]:
from src import train
metrics = train.main()
metrics

## Inference + Evidence Dossier
Run the trained auditor on the adversarial set and inspect a dossier. Every `feature_evidence` value is traceable to a real ticket field (validated).

In [ ]:
import predict
adv = pd.read_csv(C.DATA_DIR / 'adversarial_tickets.csv')
out, dossiers = predict.run_inference(adv)
out[[C.COL_ID, C.COL_PRIORITY, 'inferred_level', 'pred_mismatch', 'mismatch_type']]

In [ ]:
print(json.dumps(dossiers[0], indent=2))

## Adversarial robustness score
10 tickets engineered to fool keyword systems; ≥ 7/10 earns the bonus.

In [ ]:
from src import adversarial
report = adversarial.main()